# Customer Segmentation with K-Means (RFM)

**Author:** Tadaishe Maumbe  
**Goal:** Cluster e-commerce customers into actionable personas so the marketing team can tailor campaigns.

**Plan**
1. Generate realistic synthetic transaction data
2. Engineer RFM features (Recency, Frequency, Monetary)
3. Scale features and pick the optimal k
4. Fit K-Means
5. Visualise with PCA
6. Profile and name each persona
7. Translate into business actions

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RNG = 42
np.random.seed(RNG)

## 1. Generate synthetic transactions

In [ ]:
def generate_transactions(n_customers=5000, snapshot_date=None, seed=RNG):
    rng = np.random.default_rng(seed)
    snapshot_date = snapshot_date or datetime(2025, 1, 1)

    # Hidden segments to mimic real-world heterogeneity
    segments = rng.choice(
        ['loyal', 'regular', 'one_time', 'lapsed', 'new'],
        size=n_customers,
        p=[0.12, 0.31, 0.27, 0.18, 0.12]
    )

    rows = []
    for cid, seg in enumerate(segments):
        if seg == 'loyal':
            n_orders = rng.integers(15, 35)
            last_days = rng.integers(1, 20)
            avg_value = rng.normal(200, 40)
        elif seg == 'regular':
            n_orders = rng.integers(5, 14)
            last_days = rng.integers(7, 40)
            avg_value = rng.normal(150, 30)
        elif seg == 'one_time':
            n_orders = 1
            last_days = rng.integers(60, 200)
            avg_value = rng.normal(180, 60)
        elif seg == 'lapsed':
            n_orders = rng.integers(3, 8)
            last_days = rng.integers(150, 360)
            avg_value = rng.normal(155, 40)
        else:  # new
            n_orders = 1
            last_days = rng.integers(1, 14)
            avg_value = rng.normal(90, 25)

        first_order = snapshot_date - timedelta(days=int(last_days + rng.integers(0, 400)))
        order_dates = pd.date_range(first_order, snapshot_date - timedelta(days=int(last_days)),
                                     periods=n_orders)
        for od in order_dates:
            rows.append({
                'customer_id': cid,
                'order_date': od.date(),
                'order_value': max(round(rng.normal(avg_value, 25), 2), 5.0)
            })

    return pd.DataFrame(rows), snapshot_date

os.makedirs('data', exist_ok=True)
csv_path = 'data/transactions.csv'
snapshot = datetime(2025, 1, 1)
if not os.path.exists(csv_path):
    tx, snapshot = generate_transactions()
    tx.to_csv(csv_path, index=False)

tx = pd.read_csv(csv_path, parse_dates=['order_date'])
print(f'Transactions: {len(tx):,} | Customers: {tx["customer_id"].nunique():,}')
tx.head()

## 2. Build RFM features

In [ ]:
rfm = tx.groupby('customer_id').agg(
    recency=('order_date', lambda s: (snapshot - s.max()).days),
    frequency=('order_date', 'count'),
    monetary=('order_value', 'sum'),
).reset_index()
print(rfm.describe().round(2))
rfm.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    sns.histplot(rfm[col], bins=40, kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
plt.tight_layout(); plt.show()

Monetary and frequency are heavily right-skewed. We log-transform them to make distances more meaningful for K-Means.

In [ ]:
features = pd.DataFrame({
    'recency': rfm['recency'],
    'log_frequency': np.log1p(rfm['frequency']),
    'log_monetary': np.log1p(rfm['monetary']),
})
scaler = StandardScaler()
X = scaler.fit_transform(features)

## 3. Choose k (elbow + silhouette)

In [ ]:
ks = range(2, 11)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RNG).fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ks, inertias, 'o-')
axes[0].set_title('Elbow method'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[1].plot(ks, silhouettes, 'o-', color='orange')
axes[1].set_title('Silhouette score'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
plt.tight_layout(); plt.show()

best_k = ks[int(np.argmax(silhouettes))]
print(f'Best k by silhouette: {best_k}')

## 4. Fit K-Means

In [ ]:
k = 5  # business-meaningful k that aligns with elbow + silhouette
kmeans = KMeans(n_clusters=k, n_init=20, random_state=RNG).fit(X)
rfm['cluster'] = kmeans.labels_
rfm['cluster'].value_counts().sort_index()

## 5. Visualise with PCA

In [ ]:
pca = PCA(n_components=2, random_state=RNG)
coords = pca.fit_transform(X)
plt.figure(figsize=(9, 7))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=rfm['cluster'],
                palette='tab10', s=15, alpha=0.7)
plt.title('Customer clusters projected onto first 2 principal components')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(title='Cluster')
plt.show()

## 6. Profile clusters

In [ ]:
profile = rfm.groupby('cluster').agg(
    customers=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_revenue=('monetary', 'sum'),
).round(1).sort_values('avg_monetary', ascending=False)
profile['revenue_share_%'] = (profile['total_revenue'] / profile['total_revenue'].sum() * 100).round(1)
profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=rfm, x='cluster', y='recency', ax=axes[0], errorbar=None)
axes[0].set_title('Avg recency (days)')
sns.barplot(data=rfm, x='cluster', y='frequency', ax=axes[1], errorbar=None)
axes[1].set_title('Avg frequency (orders)')
sns.barplot(data=rfm, x='cluster', y='monetary', ax=axes[2], errorbar=None)
axes[2].set_title('Avg monetary ($)')
plt.tight_layout(); plt.show()

## 7. Name the personas and recommend actions

In [ ]:
# Map cluster IDs to human-readable personas based on the profile above
ranked = profile.reset_index()
persona_map = {}
for _, row in ranked.iterrows():
    cid = row['cluster']
    if row['avg_monetary'] > 3000 and row['avg_recency'] < 30:
        persona_map[cid] = 'Loyal high-value'
    elif row['avg_recency'] > 120:
        persona_map[cid] = 'At-risk / lapsed'
    elif row['avg_frequency'] <= 1.5 and row['avg_recency'] < 30:
        persona_map[cid] = 'New customers'
    elif row['avg_frequency'] <= 1.5:
        persona_map[cid] = 'One-time buyers'
    else:
        persona_map[cid] = 'Regulars'

rfm['persona'] = rfm['cluster'].map(persona_map)
rfm.groupby('persona').agg(
    customers=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).round(1)

## 8. Business actions per persona

| Persona | Action |
|---|---|
| **Loyal high-value** | VIP programme, early access, referral incentives. *Protect* — these customers carry the P&L. |
| **Regulars** | Cross-sell adjacent categories, loyalty tier progression. |
| **New customers** | 30-day onboarding journey with educational content and a second-purchase voucher. |
| **One-time buyers** | Re-engagement series triggered by purchase anniversary. |
| **At-risk / lapsed** | Win-back campaign with a clear, time-bound discount; suppress after 2 unsuccessful attempts to protect margin. |

**Next steps:**
- Plug RFM features into the data warehouse and refresh clusters monthly.
- A/B test the win-back offer size on the lapsed segment.
- Extend with demographic and product-category features for a richer segmentation.

---
*Built by Tadaishe Maumbe.*